# Kathmandu traffic vision - plate reader run 3

Run 3 of the plate reader's **character classifier**, adding real Kathmandu plates: 188 photos read by AI (P0151-P0400, none of them a test photo or a repeat of a test vehicle), split into characters and matched to the character detector's boxes. Then run 2 and run 3 are compared end to end on the test photos, against provisional labels (88 plates where two AI readers agree).

In [ ]:
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
"""Read a number plate: plate crop -> character boxes (chars_best.pt) -> one label per box
(glyphs_best.pt) -> text in reading order. Models come from the train-plate-reader notebook.

    python reader.py plate.jpg [--chars chars_best.pt] [--glyphs glyphs_best.pt]
"""
from __future__ import annotations

import argparse
import re

import cv2

Box = tuple  # (x1, y1, x2, y2, label, conf)


def reading_order(boxes: list[Box]) -> list[Box]:
    """Return the character boxes in the order a person reads the plate.

    Nepali plates come in one row (most car plates) or two rows (most motorcycle plates, and the
    upper line of zone + lot over the number on older red plates). Read the top row left to right,
    then the bottom row left to right.
    """
    # New row where the gap between consecutive vertical centres exceeds half a character height:
    # scales with plate size, and a tilted row's small steady steps never reach it.
    heights = sorted(b[3] - b[1] for b in boxes)
    gap = heights[len(heights) // 2] / 2
    rows, last = [], None
    for b in sorted(boxes, key=lambda b: (b[1] + b[3]) / 2):
        c = (b[1] + b[3]) / 2
        if last is None or c - last > gap:
            rows.append([])
        rows[-1].append(b)
        last = c
    return [b for row in rows for b in sorted(row, key=lambda b: b[0])]


DEV_DIGITS = set("०१२३४५६७८९")
EMB_DIGITS = set("0123456789")
DEV_PLATE = re.compile(r"L+D+L+D{1,4}")   # zone (or province), lot, vehicle class, number: बा २ च १२३४

# shortcut: code lists written from memory of Nepali plate formats; confirm against the Department of
# Transport Management list before relying on the check for enforcement
ZONES = ["मे", "को", "स", "ज", "बा", "ना", "ग", "लु", "ध", "रा", "भे", "क", "से", "म"]   # 14 old zones
PROVINCES = ["को", "म", "बा", "ग", "लु", "क", "सु",                                    # abbreviations
             "कोशी", "मधेश", "बागमती", "गण्डकी", "लुम्बिनी", "कर्णाली", "सुदूरपश्चिम"]    # full names


def valid_prefix(text: str) -> bool:
    """Does a Devanagari reading start with a real zone or province code, followed by the lot digits?

    Accepted: zone + digit (बा२...), "प्रदेश" + digit (प्रदेश३...), "प्र" + digit (प्र३...),
    province + "प्रदेश" (बागमतीप्रदेश..., बाप्रदेश...), province abbreviation + "प्र" + digit (बाप्र०६...).
    """
    starts_digit = lambda rest: rest[:1] in DEV_DIGITS
    for p in ("प्रदेश", "प्र"):
        if text.startswith(p) and starts_digit(text[len(p):]):
            return True
    for p in PROVINCES:
        if text.startswith(p + "प्रदेश") or (text.startswith(p + "प्र") and starts_digit(text[len(p) + len("प्र"):])):
            return True
    return any(text.startswith(z) and starts_digit(text[len(z):]) for z in ZONES)


def check_plate(labels: list[str], conf: float, min_conf: float = 0.6) -> tuple[bool, str]:
    """Is this reading plausible enough to use without a person checking it? (ok, reason)

    Catches structure errors: missed or extra characters, a letter in a number slot, mixed scripts,
    and unsure characters. It cannot catch a look-alike swapped for another of the same kind
    (३ for ६, ना for बा): those only show up as low confidence, if at all.
    """
    if not labels:
        return False, "no characters read"
    if conf < min_conf:
        return False, f"unsure character (confidence {conf:.2f} < {min_conf})"
    dev = [lab for lab in labels if not (lab in EMB_DIGITS or lab.isascii())]
    if dev and len(dev) < len(labels):
        return False, "mixes Devanagari and embossed characters"
    if dev:
        shape = "".join("D" if lab in DEV_DIGITS else "L" for lab in labels)
        if not DEV_PLATE.fullmatch(shape):
            return False, f"not a Nepali plate layout (letters/digits: {shape})"
        if not valid_prefix("".join(labels)):
            return False, f"unknown zone or province code at the start ({''.join(labels)[:4]})"
        return True, "ok"
    # shortcut: embossed plates are only checked for digits and letters both being present;
    # add their exact layout once it is confirmed against real plates
    if not any(lab in EMB_DIGITS for lab in labels) or all(lab in EMB_DIGITS for lab in labels):
        return False, "embossed reading without both letters and digits"
    return True, "ok"


def read_plate(plate, chars, glyphs, min_conf: float = 0.25) -> tuple[list[str], float]:
    """Character labels in reading order, and the lowest confidence (a weak link makes the whole reading weak)."""
    boxes = []
    for x1, y1, x2, y2 in chars.predict(plate, imgsz=416, conf=min_conf, verbose=False)[0].boxes.xyxy.tolist():
        crop = plate[int(y1):int(y2), int(x1):int(x2)]
        if crop.size == 0:
            continue
        r = glyphs.predict(crop, imgsz=64, verbose=False)[0]
        label = r.names[r.probs.top1]
        if label != "flag":
            boxes.append((x1, y1, x2, y2, label.removeprefix("emb_"), float(r.probs.top1conf)))
    if not boxes:
        return [], 0.0
    ordered = reading_order(boxes)
    return [b[4] for b in ordered], min(b[5] for b in ordered)



In [ ]:

import os, random, yaml
from pathlib import Path
IN, OUT = Path("/kaggle/input/datasets"), Path("/kaggle/working/data")
VAL_LISTS = {}

def label_of(img):
    parts = list(img.parts)
    if "images" in parts:                                       # YOLO layout: .../images/... -> .../labels/...
        i = len(parts) - 1 - parts[::-1].index("images")
        p = Path(*parts[:i], "labels", *parts[i + 1:]).with_suffix(".txt")
        if p.exists():
            return p
    return img.with_suffix(".txt")                              # some sets keep labels next to images

def add(tag, split, images, names, mapping, repeat=1):
    """Link images into OUT/<split>, rewriting labels: keep only source classes in `mapping`
    (source name -> target id); boxes of other classes are dropped, the image stays."""
    d = OUT / split
    (d / "images").mkdir(parents=True, exist_ok=True)
    (d / "labels").mkdir(parents=True, exist_ok=True)
    n_box, kept = 0, []
    for img in images:
        rows, lab = [], label_of(img)
        if lab.exists():
            for ln in lab.read_text(errors="ignore").splitlines():
                parts = ln.split()
                if len(parts) != 5 or not parts[0].isdigit():      # skip polygons / malformed rows
                    continue
                k = int(parts[0])
                if k < len(names) and names[k] in mapping:
                    rows.append(" ".join([str(mapping[names[k]])] + parts[1:]))
        for r in range(repeat):
            stem = f"{tag}_{r}_{img.parent.name}_{img.stem}".replace(" ", "_")
            link = d / "images" / (stem + img.suffix.lower())
            if not link.exists():
                link.symlink_to(img)
            (d / "labels" / (stem + ".txt")).write_text("\n".join(rows))
            kept.append(str(link))
        n_box += len(rows)
    if split == "val":
        VAL_LISTS[tag] = kept
    print(f"{tag:<10} {split:<5} {len(images):>6} images x{repeat}  {n_box:>7} boxes kept")

def imgs(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

def split_random(images, val_share=0.15, seed=0):
    images = images[:]
    random.Random(seed).shuffle(images)
    n = int(len(images) * val_share)
    return images[n:], images[:n]

def write_yaml(names):
    (OUT / "data.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": "val/images",
                                                 "names": dict(enumerate(names))}))
    for tag, files in VAL_LISTS.items():          # per-source validation sets, to see where the model is weak
        (OUT / f"val_{tag}.txt").write_text("\n".join(files))
        (OUT / f"data_{tag}.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images",
                                                            "val": f"val_{tag}.txt", "names": dict(enumerate(names))}))


In [ ]:

import random, shutil, glob
from collections import Counter
from pathlib import Path
from ultralytics import YOLO
CLS = Path("/kaggle/working/glyphs")
VAL = {}     # source -> [(image link, class)]

def link_all(tag, pairs, split):
    for i, (img, c) in enumerate(pairs):
        d = CLS / split / c
        d.mkdir(parents=True, exist_ok=True)
        dst = d / f"{tag}_{i}{img.suffix.lower()}"
        if not dst.exists():
            dst.symlink_to(img)
        if split == "val":
            VAL.setdefault(tag, []).append((dst, c))
    print(f"{tag:<10} {split:<5} {len(pairs):>6} crops, {len(set(c for _, c in pairs))} classes")

# Devanagari: one folder per character, no split given -> random 15% validation
dev = IN / "inspiring-lab/nepali-number-plate-characters-dataset/character_ocr"
pairs = [(p, p.parent.name) for p in imgs(dev)]
random.Random(0).shuffle(pairs)
n = int(len(pairs) * 0.15)
link_all("devanagari", pairs[n:], "train")
link_all("devanagari", pairs[:n], "val")

# Embossed: class folders under train/ (2,670), valid/ (349) and test/ (563); the flat augmented/ folder
# (6,566 files, no class in path or name) is skipped. utsavbaral007's set has the same test folders: a copy, not used.
emb = IN / "theciceerguy/embossed-number-plate-detection-nepal-dataset/digits_and_numbers_dataset"
name = lambda p: "flag" if p.parent.name == "Nepali Flag" else "emb_" + p.parent.name
link_all("embossed", [(p, name(p)) for p in imgs(emb / "train")], "train")
link_all("embossed", [(p, name(p)) for p in imgs(emb / "valid") + imgs(emb / "test")], "val")


from collections import Counter
ROMAN = {**{str(i): "०१२३४५६७८९"[i] for i in range(10)},
         "ba": "बा", "pa": "प", "cha": "च", "kha": "ख", "ko": "को", "me": "मे", "pra": "प्र"}
NEW = {"ishwor_chars": IN / "ishworsubedii/nepalese-license-plate-character-dataset/dataset",
       "kshitiz_chars": IN / "kshitizgajurel042/individual-character-images-of-nepali-lp/NepaliLPCharacters"}
run1 = YOLO([p for p in sorted(glob.glob("/kaggle/input/**/glyphs_best.pt", recursive=True)) if "run-2" not in p][0])
unseen = []
for tag, root in NEW.items():
    ok_n = all_n = 0
    pairs = []
    for folder in sorted(p for p in root.iterdir() if p.is_dir()):
        files = imgs(folder)
        want = ROMAN.get(folder.name)
        got = Counter()
        for i in range(0, len(files), 512):
            for r in run1.predict([str(f) for f in files[i:i + 512]], imgsz=64, verbose=False):
                got[r.names[r.probs.top1]] += 1
        top, n = got.most_common(1)[0]
        hit = got.get(want, 0)
        print(f"{tag:<14} {folder.name:<4} -> {want}  {len(files):>4} crops, run-1 says {top} for {n}, {hit / len(files):.1%} agree")
        if want is not None:            # every mapped folder counts, kept or dropped, so the score is not flattered
            ok_n += hit
            all_n += len(files)
        if want is None or top != want:
            print(f"   dropped: mapping not confirmed")
            continue
        pairs += [(f, want) for f in files]
    unseen.append({"source": tag, "crops": all_n, "run1_top1_unseen": round(ok_n / all_n, 4)})
    random.Random(0).shuffle(pairs)
    n = int(len(pairs) * 0.15)
    link_all(tag, pairs[n:], "train")
    link_all(tag, pairs[:n], "val")
import pandas as pd
pd.DataFrame(unseen).to_csv("/kaggle/working/run1_on_unseen_sources.csv", index=False)
print(pd.DataFrame(unseen).to_string(index=False))

import zipfile, re, cv2, numpy as np
find = lambda name: sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
plates_m = YOLO(find("plates_best.pt")[0])
chars_m = YOLO(find("chars_best.pt")[0])
run2 = YOLO([p for p in find("glyphs_best.pt") if "run-2" in p][0])
PH = Path("/kaggle/working/photos"); zipfile.ZipFile(find("photos.zip")[0]).extractall(PH)
read_tsv = lambda f: {l.split("\t")[0]: l.split("\t")[1] for l in open(f, encoding="utf-8").read().splitlines() if "\t" in l}
LAB = read_tsv(find("labels_claude_train.txt")[0])
MARKS = {chr(c) for c in [*range(0x0900, 0x0904), *range(0x093A, 0x094D), 0x094E, 0x094F, 0x0962, 0x0963]}

def units(text):
    # one unit per written character: consonant (+ virama + consonant)* + vowel signs, e.g. प्रदेश -> प्र, दे, श
    out = []
    for ch in text:
        if out and (ch in MARKS or ch == "्" or out[-1].endswith("्")):
            out[-1] += ch
        else:
            out.append(ch)
    return out
assert units("बागमतीप्रदेश०६") == ["बा", "ग", "म", "ती", "प्र", "दे", "श", "०", "६"]

def plate_crop(img):
    b = plates_m.predict(img, imgsz=1280, conf=0.2, verbose=False)[0].boxes.xyxy.tolist()
    if not b:
        return None
    h, w = img.shape[:2]   # union: the detector splits close-up plates into pieces
    x1, y1, x2, y2 = min(v[0] for v in b), min(v[1] for v in b), max(v[2] for v in b), max(v[3] for v in b)
    px, py = (x2 - x1) * 0.08, (y2 - y1) * 0.15
    return img[int(max(0, y1 - py)):int(min(h, y2 + py)), int(max(0, x1 - px)):int(min(w, x2 + px))]

OUTR = Path("/kaggle/working/real"); OUTR.mkdir(exist_ok=True)
stats, by_photo = Counter(), {}
known = set(run2.names.values())
for pid, text in LAB.items():
    crop = plate_crop(cv2.imread(str(PH / f"{pid}.jpg")))
    if crop is None:
        stats["no plate found"] += 1; continue
    boxes = [(x1, y1, x2, y2, "", 1.0) for x1, y1, x2, y2 in
             chars_m.predict(crop, imgsz=416, conf=0.25, verbose=False)[0].boxes.xyxy.tolist()]
    u = units(text)
    if len(boxes) != len(u):
        stats["character count differs"] += 1; continue
    boxes = reading_order(boxes)
    cs = [crop[int(b[1]):int(b[3]), int(b[0]):int(b[2])] for b in boxes]
    if any(c.size == 0 for c in cs):
        stats["empty crop"] += 1; continue
    pred = [r.names[r.probs.top1].removeprefix("emb_") for r in run2.predict(cs, imgsz=64, verbose=False)]
    k = [(p, x) for p, x in zip(pred, u) if x in known]
    if k and sum(p == x for p, x in k) < 0.5 * len(k):
        stats["looks shifted"] += 1; continue
    files = []
    for j, (c, x) in enumerate(zip(cs, u)):
        f = OUTR / f"{pid}_{j}.jpg"; cv2.imwrite(str(f), c); files.append((f, x))
    by_photo[pid] = files; stats["used"] += 1
print("real plates:", dict(stats))
ids = sorted(by_photo); random.Random(0).shuffle(ids); nv = max(1, len(ids) * 15 // 100)
tr = [p for k in ids[nv:] for p in by_photo[k]]           # split by photo: no plate on both sides
va = [p for k in ids[:nv] for p in by_photo[k]]
link_all("kathmandu_real", tr * 3, "train")                 # few but closest to real use: weighted x3
link_all("kathmandu_real", va, "val")
print("new classes:", sorted({x for _, x in tr + va} - known))

cls = YOLO("yolo11s-cls.pt")
cls.train(data=str(CLS), epochs=40, time=1.5, imgsz=64, batch=256, patience=10,
          project="/kaggle/working/runs", name="glyphs", workers=4, seed=0, plots=True)
shutil.copy(sorted(glob.glob("/kaggle/working/runs/glyphs*/weights/best.pt"))[-1], "/kaggle/working/glyphs_best.pt")

m = YOLO("/kaggle/working/glyphs_best.pt")
rows = []
for tag, items in VAL.items():
    wrong = Counter()
    ok = 0
    for i in range(0, len(items), 512):
        batch = items[i:i + 512]
        for (img, c), r in zip(batch, m.predict([str(p) for p, _ in batch], imgsz=64, verbose=False)):
            got = r.names[r.probs.top1]
            ok += got == c
            if got != c:
                wrong[f"{c} -> {got}"] += 1
    rows.append({"source": tag, "crops": len(items), "top1": round(ok / len(items), 4),
                 "most_confused": "; ".join(f"{k} ({v})" for k, v in wrong.most_common(5))})
import pandas as pd
res = pd.DataFrame(rows)
res.to_csv("/kaggle/working/glyphs_per_source.csv", index=False)
print(res.to_string(index=False))
shutil.rmtree(CLS)


In [ ]:

# End-to-end on the test photos (never trained on). Labels are PROVISIONAL: the 88 plates where Gemini and Claude
# read the same text; the owner's hand labels replace them later.
import pandas as pd
TEST = read_tsv(find("test_provisional.tsv")[0])
run3 = YOLO("/kaggle/working/glyphs_best.pt")
digits = lambda s: (re.findall(r"[०-९]+$", s) or [""])[0]
rows = []
for pid, truth in sorted(TEST.items()):
    crop = plate_crop(cv2.imread(str(PH / f"{pid}.jpg")))
    r = {"id": pid, "truth": truth}
    for name, g in [("run2", run2), ("run3", run3)]:
        labels, cf = read_plate(crop, chars_m, g) if crop is not None else ([], 0.0)
        r[name] = "".join(labels); r[name + "_check"] = check_plate(labels, cf)[1]
    rows.append(r)
df = pd.DataFrame(rows); df.to_csv("/kaggle/working/end2end_provisional.csv", index=False)
for name in ("run2", "run3"):
    exact = (df[name] == df.truth).mean(); num = (df[name].map(digits) == df.truth.map(digits)).mean()
    ok = df[name + "_check"] == "ok"; okexact = (df[name] == df.truth)[ok].mean() if ok.any() else float("nan")
    print(f"{name}: whole plate exact {exact:.1%}, number part exact {num:.1%}, passes check {ok.mean():.1%}, "
          f"exact among those passing {okexact:.1%}  (n={len(df)} provisional)")
